In [1]:
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

import os
import sys

# use absolute path here
project_path = "/mnt/School/PhD/AI221/Project/"
sys.path.insert(0, project_path)

from src.data_extraction.utils.raster import extract_raster_stats
from src.data_extraction.utils.constants import data_path

In [2]:
twi_path = os.path.join(data_path, "ph_cop30_tiles/twi_per_city")

In [3]:
ph_bounds = gpd.read_file(os.path.join(data_path,"ph_adm3_municities/PH_Adm3_MuniCities.shp.shp"))
ph_bounds = ph_bounds[ph_bounds["geo_level"] == "City"]
ph_bounds.head()

,adm1_psgc,adm2_psgc,adm3_psgc,adm3_en,geo_level,len_crs,area_crs,len_km,area_km2,geometry
4,100000000,102800000,102805000,City of Batac,City,66661,158252391,66,158.0,"POLYGON ((247341.309 2003933.537, 247293.327 2..."
11,100000000,102800000,102812000,City of Laoag,City,53964,110146974,53,110.0,"POLYGON ((248393.247 2016552.78, 248424.831 20..."
28,100000000,102900000,102906000,City of Candon,City,62247,77652664,62,77.0,"POLYGON ((230319.064 1907309.065, 230338.289 1..."
56,100000000,102900000,102934000,City of Vigan,City,25067,24485368,25,24.0,"POLYGON ((221597.447 1945779.016, 221718.087 1..."
70,100000000,103300000,103314000,City of San Fernando,City,54233,99006121,54,99.0,"POLYGON ((225191.401 1841887.86, 225339.01 184..."


In [4]:
twi_cols = ["adm3_en", "adm3_psgc", "mean_twi", "max_twi", "std_twi"]
twi_gdfs = []

for psgc in ph_bounds["adm3_psgc"].unique():
    psgc_path = os.path.join(twi_path, f"{psgc}_final_twi.tif")
    twi_gdf =  extract_raster_stats(
        ph_bounds[ph_bounds["adm3_psgc"] == psgc],
        psgc_path,
        "twi", 
    )
    twi_gdfs.append(twi_gdf[twi_cols])

twi_final_gdf = pd.concat(twi_gdfs, axis=0, ignore_index=True)
twi_final_gdf.head()

Processing 102805000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 102812000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 102906000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 102934000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 103314000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 105503000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 105518000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 105532000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 105546000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 201529000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 203108000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 203114000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 203135000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Processing 300803000_final_twi.tif...
 -> CRS: EPSG:4326 | NoData: 0.0
Proces

,adm3_en,adm3_psgc,mean_twi,max_twi,std_twi
0,City of Batac,102805000,-3.754234,20.971954,4.242962
1,City of Laoag,102812000,-2.450579,20.181175,4.329910
2,City of Candon,102906000,-3.141669,20.913902,3.924423
3,City of Vigan,102934000,-2.407162,30.716255,3.613563
4,City of San Fernando,103314000,-4.519191,33.818222,3.077760


In [5]:
# Validation: All cities in the ph_bounds gdf should have TWI metrics
validation_df = pd.merge(
    ph_bounds[["adm3_psgc"]],
    twi_final_gdf,
    how = "outer",
    indicator=True
)
validation_df[validation_df["_merge"] != "both"]

,adm3_psgc,adm3_en,mean_twi,max_twi,std_twi,_merge


In [7]:
twi_final_gdf.to_csv(
    os.path.join(data_path, "urban_twi_metrics.csv"), 
    index=False
)

## Validation

In [8]:
from pathlib import Path

import numpy as np
import rasterio

expected = ph_bounds[["adm3_psgc", "adm3_en"]].copy()
expected["adm3_psgc"] = expected["adm3_psgc"].astype(str)

metrics_df = pd.read_csv(os.path.join(data_path, "urban_twi_metrics.csv"))
metrics_df["adm3_psgc"] = metrics_df["adm3_psgc"].astype(str)

out_dir = Path(twi_path)

records = []

for _, row in expected.iterrows():
    psgc = row["adm3_psgc"]
    city_name = row["adm3_en"]
    tif_path = out_dir / f"{psgc}_final_twi.tif"

    rec = {
        "adm3_psgc": psgc,
        "adm3_en": city_name,
        "file_exists": tif_path.exists(),
        "opens": False,
        "width": None,
        "height": None,
        "crs": None,
        "nodata": None,
        "valid_pixel_count": None,
        "all_nodata": None,
        "metrics_row_exists": False,
        "mean_twi": None,
        "max_twi": None,
        "std_twi": None,
        "null_metric": None,
        "bad_std": None,
        "error": None,
    }

    # raster checks
    if tif_path.exists():
        try:
            with rasterio.open(tif_path) as src:
                rec["opens"] = True
                rec["width"] = src.width
                rec["height"] = src.height
                rec["crs"] = str(src.crs) if src.crs else None
                rec["nodata"] = src.nodata

                arr = src.read(1, masked=True)
                valid_pixel_count = int(np.ma.count(arr))
                rec["valid_pixel_count"] = valid_pixel_count
                rec["all_nodata"] = valid_pixel_count == 0
        except Exception as e:
            rec["error"] = str(e)

    # metrics checks
    metric_match = metrics_df.loc[metrics_df["adm3_psgc"] == psgc]
    if len(metric_match) > 0:
        rec["metrics_row_exists"] = True
        rec["mean_twi"] = metric_match["mean_twi"].iloc[0]
        rec["max_twi"] = metric_match["max_twi"].iloc[0]
        rec["std_twi"] = metric_match["std_twi"].iloc[0]

        rec["null_metric"] = metric_match[["mean_twi", "max_twi", "std_twi"]].iloc[0].isna().any()
        rec["bad_std"] = pd.notna(rec["std_twi"]) and rec["std_twi"] < 0

    records.append(rec)

validation_df = pd.DataFrame(records)

validation_df["nonzero_shape"] = (
    validation_df["width"].fillna(0).gt(0) & validation_df["height"].fillna(0).gt(0)
)

validation_df["passes_validation"] = (
    validation_df["file_exists"]
    & validation_df["opens"]
    & validation_df["nonzero_shape"]
    & ~validation_df["all_nodata"].fillna(True)
    & validation_df["metrics_row_exists"]
    & ~validation_df["null_metric"].fillna(True)
    & ~validation_df["bad_std"].fillna(True)
)

summary = {
    "expected_cities": len(expected),
    "files_found": int(validation_df["file_exists"].sum()),
    "files_opened": int(validation_df["opens"].sum()),
    "metrics_rows_found": int(validation_df["metrics_row_exists"].sum()),
    "passes_validation": int(validation_df["passes_validation"].sum()),
    "problems": int((~validation_df["passes_validation"]).sum()),
}

print(summary)

problems_df = validation_df.loc[~validation_df["passes_validation"]].copy()
problems_df = problems_df.sort_values(["adm3_psgc"])

display(
    problems_df[
        [
            "adm3_psgc",
            "adm3_en",
            "file_exists",
            "opens",
            "nonzero_shape",
            "all_nodata",
            "metrics_row_exists",
            "null_metric",
            "bad_std",
            "error",
        ]
    ]
)

{'expected_cities': 149, 'files_found': 149, 'files_opened': 149, 'metrics_rows_found': 149, 'passes_validation': 149, 'problems': 0}


,adm3_psgc,adm3_en,file_exists,opens,nonzero_shape,all_nodata,metrics_row_exists,null_metric,bad_std,error
